In [1]:
import hoda
import tensorly as tl

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==1.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation



paradigm = P300(resample=64)
datasets = [
       #BI2012(),
       #BI2014a(),
       #BI2014b(),
       #BI2015a(),
       #BI2015b(),
       BNCI2014008(),
       #BNCI2014009(),
       #BNCI2015_003(),
       #Cattan2019_VR(),
       #EPFLP300(),
       #Huebner2017(),
       #Huebner2018(),
       #Lee2019_ERP(),
       #Sosulski2019()   
   ]



<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
/usr/local/lib/python3.10/dist-packages/moabb/pipelines/__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(
BNCI2014008 has been renamed to BNCI2014_008. BNCI2014008 will be removed in version 1.1.
The dataset class name 'BNCI2014008' must be an abb

To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.


In [5]:
from sklearn.pipeline import Pipeline
from mne.decoding import Scaler
from hoda.hoda import  HODA, GreedyBTTDA
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, GridSearchCV

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hoda_params = dict(
    max_iter=128,
    tol=1e-12,
    init ='random',
    shrinkage='lw',
    toeplitz=None,
    obj='rt',
    ortho=False,
    solver='lanczos',
    taper=False,
    extra_train_info=False,
    verbose=False, 
)

bttda = GreedyBTTDA(
        ranks=[None]*8,
        hoda_params=hoda_params,
        verbose=True,
        extra_train_info=False,
)

In [6]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import dask


%env PYTHONWARNINGS=ignore

df = []
for dataset in datasets:
    X_sub, y_sub, meta = paradigm.get_data(dataset, 
        subjects=[4]
    )
    idc = meta.reset_index()\
        .groupby(['subject', 'session'])\
        .index.aggregate(list)
    
    for session, session_idc in idc.items():
        print(dataset.code + ' ' + str(session))
        session_idc = np.array(session_idc) 

        
        X =X_sub[session_idc]
        y = y_sub[session_idc]
        
        X = tl.tensor(X)

        X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=test_size, shuffle=True, random_state=42)

        bttda.fit(X_train,y_train, X_test=X_test, y_test=y_test)
        df_session = bttda.train_info_
        df_session['dataset'] = dataset.code
        df_session['subject'] = session[0]
        df_session['session'] = session[1]
        df.append(df_session)
df = pd.concat([pd.DataFrame(d) for d in df], axis=0)

env: PYTHONWARNINGS=ignore
MNE_DATA is not already configured. It will be set to default location in the home directory - /root/mne_data
All datasets will be downloaded to this location, if anything is already downloaded, please move manually to this location


/usr/local/lib/python3.10/dist-packages/moabb/datasets/download.py:55: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))


BNCI2014-008 (4, '0')


TypeError: BTTDA.fit() got an unexpected keyword argument 'X_test'

In [ ]:
pd.set_option("display.max_rows", None)
df

In [ ]:
import math
df['n_features'] = df['rank'].apply(math.prod)
df['n_features'] = df.groupby(['dataset', 'subject', 'session'])['n_features'].cumsum()
df

In [ ]:
df_diff = df.reset_index()
idx = ['dataset', 'subject', 'session']
df_diff = df_diff.set_index(idx)

df_test_score_first = df_diff.groupby(['dataset', 'subject', 'session']).test_score.aggregate('first')
df_diff['test_score_diff_hoda'] = df_diff['test_score'] - df_test_score_first
df_val_score_first = df_diff.groupby(['dataset', 'subject', 'session']).val_score.aggregate('first')
df_diff['val_score_diff_hoda'] = df_diff['val_score'] - df_val_score_first

df_diff = df_diff.reset_index()
df_diff

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


sns.relplot(
    data=df_diff,
    col='dataset',
    #col_wrap=4,
    x='block',
    y='test_score_diff_hoda',
    hue='subject',
    kind="line",
    palette="tab10",
    #ci=None,
    #units='fold',
    #estimator=None,
)
plt.axhline(0, color='black')
plt.show()



In [ ]:
sns.relplot(
    data=df_diff,
    col='dataset',
    #col_wrap=4,
    x='block',
    y='val_score_diff_hoda',
    hue='subject',
    kind="line",
    palette="tab10",
    #ci=None,
    #units='fold',
    #estimator=None,
)
plt.show()

In [ ]:
sns.relplot(
    data=df_diff,
    col='dataset',
    #col_wrap=4,
    x='block',
    y='test_score',
    hue='session',
    kind="line",
    palette="tab10",
    #ci=None,
    #units='fold',
    #estimator=None,
)
plt.show()